# V-JEPA Gym 训练启动板 (Google Colab)

此 Notebook 提供一键执行环境。它将：
1. 拉取你的 GitHub 仓库
2. 安装 MuJoCo 等相关依赖，并配置无头渲染环境
3. 运行端到端训练与评估 Pipeline

In [ ]:
# 1. 克隆你的 GitHub 项目
# ⚠️ 请将 YOUR_USERNAME/YOUR_REPO_NAME 替换为你实际的 GitHub 地址
REPO_URL = "https://github.com/OopsYouDiedE/Tao-43.git"
!git clone $REPO_URL

import os
repo_name = REPO_URL.split('/')[-1].replace('.git', '')
os.chdir(repo_name)
print(f"已切换到工作目录: {os.getcwd()}")

In [ ]:
# 2. 安装环境与系统依赖 (MuJoCo EGL 渲染必备)
!apt-get update -y
!apt-get install -y libgl1-mesa-dev libgl1-mesa-glx libglew-dev libosmesa6-dev patchelf
!pip install dm_control h5py imageio[ffmpeg]

# 强制 MuJoCo 使用 EGL 后端进行无显示器渲染
os.environ['MUJOCO_GL'] = 'egl'

import torch
print(f"\n环境配置完成！当前使用的 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '未检测到 GPU，请在 修改 -> 笔记本设置 中开启 T4 GPU'}")

In [ ]:
# 3. 验证 V-JEPA 官方模型加载及环境有效性
!python main.py verify --require-official-vjepa

In [ ]:
# 4. 运行神谕基准测试 (Oracle Demo) - 可选，测试 MuJoCo 渲染是否正常工作
!python main.py oracle-demo --mpc-steps 15

In [ ]:
# 5. 一键端到端训练与评估 Pipeline
# 包含: collect-synthetic -> collect-latent-shards -> train-ac-latent -> eval-ac
!python main.py pipeline-latent --train-windows 5000 --val-windows 500 --epochs 5 --eval-mpc-steps 15

In [ ]:
# 6. 查看评估生成的演示视频
from IPython.display import HTML
from base64 import b64encode

def show_video(video_path):
    if not os.path.exists(video_path):
        return HTML(f"<b style='color:red'>找不到视频文件: {video_path}</b>")
    mp4 = open(video_path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=400 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """)

show_video("runs/pipeline_latent/latest/eval_outputs/execution.mp4")